In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

from week1.scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
api_key=os.getenv('OPENROUTER_API_KEY')
model='gpt-5-nano'
openrouter=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=api_key)


In [4]:
links_system_prompt="""You are an expert at analyzing website navigation structures to identify the most relevant pages for company brochures and marketing materials.

Your task is to evaluate a list of links from a company website and select only those that would be valuable for understanding the company's business, culture, and opportunities.

RELEVANT LINK TYPES (include these):
- About/About Us pages
- Company information or history pages
- Team/Leadership pages
- Careers/Jobs/Work With Us pages
- Mission/Values pages
- Products/Services overview pages
- News/Press/Media pages
- Case Studies/Portfolio/Work pages
- Contact pages (but not email links)
- Investor Relations pages

EXCLUDE (do not include these):
- Privacy Policy, Terms of Service, Cookie Policy, Legal notices
- Direct email links (mailto:)
- Social media links
- Login/Account pages
- Shopping cart or checkout pages
- Search functionality
- Language selectors
- Duplicate links to the same page
- Footer navigation that duplicates main navigation
- Downloads of individual files (unless they're annual reports or key company documents)

OUTPUT FORMAT:
Return a JSON object with:
1. A "links" array containing selected links
2. Each link should have: "type" (descriptive category), "url" (complete https URL)

Example response:
{
    "links": [
        {
            "type": "about page",
            "url": "https://full.url/goes/here/about"
        },
        {
            "type": "careers page",
            "url": "https://another.full.url/careers"
        }
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt="""Analyze the following links from the website: {url}

Your task:
1. Identify which links are most relevant for a company brochure
2. Convert any relative URLs to complete https:// URLs using the base domain: {url}
3. Eliminate duplicates (same destination, different link text)
4. Exclude privacy policies, terms of service, and email links
5. Assign relevance scores based on importance for understanding the company
6. Return results in the specified JSON format, sorted by relevance


Links found on the page:


"""
    web=fetch_website_links(url)
    user_prompt+="\n".join(web)
    return user_prompt

In [18]:
print(get_links_user_prompt('https://delgusto-bh.com'))

Analyze the following links from the website: {url}

Your task:
1. Identify which links are most relevant for a company brochure
2. Convert any relative URLs to complete https:// URLs using the base domain: {url}
3. Eliminate duplicates (same destination, different link text)
4. Exclude privacy policies, terms of service, and email links
5. Assign relevance scores based on importance for understanding the company
6. Return results in the specified JSON format, sorted by relevance


Links found on the page:


#qodef-page-content
https://delgusto-bh.com/
https://delgusto-bh.com/menu-vertical/
https://delgusto-bh.com/shop/
https://delgusto-bh.com/about-us-2/
https://delgusto-bh.com/contact/
https://delgusto-bh.com/
https://fidalgo.qodeinteractive.com/book-a-table/
https://delgusto-bh.com/
https://delgusto-bh.com/menu-vertical/
https://delgusto-bh.com/shop/
https://delgusto-bh.com/about-us-2/
https://delgusto-bh.com/contact/
https://delgusto-bh.com/
https://fidalgo.qodeinteractive.com/book

In [20]:
def select_relevant_links(url):
    response=openrouter.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": links_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type" : "json_object"}
    )
    result=response.choices[0].message.content
    links=json.loads(result)
    return links

In [24]:
def fetch_page_and_all_relevant_links(url):
    content=fetch_website_contents(url)
    links=select_relevant_links(url)
    result=f"content of the landing page: {content} \n"
    for link in links['links']:
        result+=f"\nthe content of {link['type']} is :\n"
        result+=fetch_website_contents(link['url'])
    return result

In [25]:
print(fetch_page_and_all_relevant_links('https://issalofficial.com'))

content of the landing page: Home - issalofficial.com

Skip to content
Home
Shop
Catalog
blog
Contact
About
0.00
د.ج
0.00
د.ج
Main Menu
Best Quality Products
Your Way To Live Healthy Life !
We provide 100% natural and healthy products to help you maintain a balanced lifestyle. Our organic food selection is free from chemicals , ensuring the best quality for you . Choose nature!
Shop Now
Certified Organic
100% Guarantee
At Your Service
24 / 7
Healthy Products
100% Natural & Healthy
Best Selling Products
Peanut Butter
1,000.00
د.ج
Add to cart
Sola Chocolat
1,000.00
د.ج
Add to cart
SOLA ORANGE
500.00
د.ج
Add to cart
Tisane
500.00
د.ج
Add to cart
Drinks
Herbal Drinks – Refreshing and full of goodness.
Shop Now
Snacks
Healthy Snacks – 100% natural, nutritious, and delicious!
Shop Now
Butters
Natural Butters – Pure, creamy, and additive-free.
Shop Now
Get Your Perfect Healthy Start Today!
Shop Now
Customer Reviews
★
★
★
★
★
mafihomch ga3 banat l7lawa zayda wala zoyout wlh fraaht bihoom bazaa

In [27]:
brochure_system_prompt = """
You are an expert marketing copywriter specializing in creating compelling company brochures.

Your task is to analyze website content and create a professional, engaging brochure that effectively
communicates the company's value proposition to three key audiences: prospective customers, investors,
and potential recruits.

BROCHURE STRUCTURE:
Create a well-organized brochure with the following sections (only include sections where you have substantial information):

1. Company Overview - Brief introduction, mission, and what the company does
2. Products/Services - Key offerings and value propositions
3. Market Position - Industry focus, target customers, notable clients/case studies
4. Company Culture & Values - Work environment, core values, team information
5. Growth & Achievements - Milestones, awards, funding, metrics (if available)
6. Career Opportunities - Open roles, benefits, why join (if available)
7. Contact Information - How to get in touch

WRITING GUIDELINES:
- Write in a professional yet engaging tone
- Use clear, concise language - avoid jargon unless industry-appropriate
- Focus on unique value propositions and differentiators
- Include specific details, metrics, and examples when available
- Highlight concrete achievements over vague claims
- Keep the total brochure to 500-800 words unless there's substantial unique information
- Use markdown formatting: headers (##), bold for emphasis, lists where appropriate
- Do NOT use code blocks - output should be clean markdown
- If information is missing for a section, skip it rather than making assumptions

TONE:
- Professional and credible
- Enthusiastic but not hyperbolic
- Fact-based and specific
- Tailored to the company's industry and audience
"""

In [28]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Create a professional company brochure for: {company_name}

Below is the content gathered from their website, including the landing page and other relevant pages.
Analyze this information carefully and create a compelling brochure that would appeal to customers,
investors, and potential employees.

INSTRUCTIONS:
- Extract key facts, value propositions, and unique differentiators
- Organize information into logical sections with clear headers
- Include specific details: metrics, client names, products, achievements, etc.
- Identify and emphasize what makes this company stand out
- If certain information is missing (e.g., careers, culture), simply omit those sections
- Ensure the brochure flows well and tells a coherent story about the company
- Output in clean markdown format without code blocks

WEBSITE CONTENT:

"""
    user_prompt+=fetch_page_and_all_relevant_links(url)
    user_prompt=user_prompt[:5000]
    return user_prompt

In [31]:
def generate_brochure(company_name,url):
    stream=openrouter.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name,url)}

        ],
        stream=True
    )
    response=""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream :
        response+=chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
generate_brochure("issal","https://issalofficial.com")

## ISSAL: Pure, Organic, Natural

ISSAL is your trusted source for pure, organic, and chemical-free foods designed to support a balanced, healthy lifestyle. From nutritious nut butters to soothing herbal teas and wholesome snacks, every product is crafted with care to bring you the best of nature. Based in Touggourt, Algeria, ISSAL is committed to quality, sustainability, and well-being—because you deserve the best that nature has to offer.

- 100% natural, chemical-free ingredients
- Certified organic mindset across all offerings
- A selection designed for everyday health and delight

## Products / Services

ISSAL’s shop brings together four core product categories, each built around clarity, quality, and convenience.

- **Butters**
  - Peanut Butter — 1,000 د.ج
  - Granola Butter — 1,000 د.ج
  - CAFFÈ D’ORZO — 500 د.ج
  - Date Sugar — 500 د.ج
  - Granola Butter (alternative) — 1,000 د.ج
- **Drinks**
  - Herbal Drinks (refreshing and wholesome)
  - Tisane — 500 د.ج
- **Snacks**
  - Healthy Snacks — 100% natural, nutritious, and delicious
  - ENERGY Balls — 1,200 د.ج
  - Granola — 1,500 د.ج
  - Granola Chocolate — 1,500 د.ج
- **Granola / Granola-based offerings**
  - Granola — 1,500 د.ج
  - Granola Butter — 1,000 د.ج
  - Granola Sale — 1,000 د.ج
  - BSISSA — 500 د.ج

Key value propositions:
- 100% natural, chemical-free ingredients
- Organic-certified products with a guaranteed quality standard
- A curated assortment designed to support healthy daily choices
- Accessible pricing across a broad range of healthy foods

Customer feedback highlights the quality and presentation: packaging is described as beautiful and classy, with many customers praising the natural taste and thoughtful branding.

## Market Position

- **Industry focus:** Organic, chemical-free foods designed for health-conscious consumers.
- **Target customers:** Individuals and families seeking pure, additive-free nutrition—snack lovers, wellness-minded shoppers, and fans of natural ingredients.
- **Credentials & commitment:** Certified Organic products with a 100% natural guarantee; “Healthy products” and “At Your Service 24/7” positioning on the storefront signaling reliability and accessibility.
- **Notable customer sentiment:** Jpg-style testimonials emphasize quality, packaging, and taste, illustrating strong consumer satisfaction and brand resonance.

What makes ISSAL stand out:
- A tightly focused portfolio of 100% natural, organic options across snacks, butters, and beverages
- A genuine emphasis on quality and sustainability, supported by clear, consumer-facing guarantees
- A growing, customer-loved catalog (plus “Numbers Speak For Themselves” metrics)

## Growth & Achievements

- **Product breadth:** +100 curated products across 3 product categories (with a fourth category expanding the lineup) demonstrates a broad, carefully selected assortment.
- **Customer trust:** +50 satisfied customers reflect early positive reception and repeat interest.
- **Category depth:** 3 core product categories (with ongoing expansion into additional offerings, including beverages, snacks, and granola variants).
- **Quality commitments:** Certified Organic products and a 100% natural guarantee reinforce trust and safety for health-focused shoppers.
- Customer feedback highlights strong packaging design and a favorable perception of branding integrity.

ISSAL positions itself as a reliable, nature-first option in the Algerian organic foods space, combining transparency, quality, and a growing product library to support healthy lifestyles.

## Contact Information

- Phone: +213 560 40 19 99
- Email: contact.issalofficial@gmail.com
- Address: Touggourt, Algeria
- Social:
  - Facebook
  - Instagram
  - Tiktok

If you’re looking for a trustworthy source of organic, additive-free foods designed to support healthy living, ISSAL offers a well-curated, quality-forward lineup with clear value and a customer-centric approach. For customers seeking reliable organic choices, investors tracking a small-but-growing natural foods brand, or potential team members who share a passion for natural goodness, ISSAL presents a coherent story of quality, sustainability, and everyday health.